# 非ARIS語の辞書作成（アンカー法）
- アンカー：ARIS（ARIS・RIS・全角・アリスケア）と「看取りパンフ」（2016年2月〜。ARIS定着前の書き方を拾う）
- 対象：transport_final.csv の1,290例、index 日までの全記録
- **Phase 1**：アンカーを含む文とその前後 `CTX_SENT` 文から n-gram を作り、アンカーのない記録の文（背景）と比べて候補語を出す → `2_candidate_terms.csv` の `adopt` 列に記入
  - 同じ患者の同じ文は1回だけ数える（コピペ対策）
  - 臨死期の徴候・薬剤・処置の語は `STOP_RE` で除く
- **Phase 2**：`adopt` = 1 の語について、アンカーのない記録から KWIC を出す（旧判定の群×時期で層別）→ `判定` 列に記入
- 旧判定（unexpected_death.unex24）は層別と出現率の参考にだけ使い、語の選択には使わない
- DBは読み取りのみ

In [ ]:
# 0. 設定
import sqlite3, re, unicodedata, random
from collections import defaultdict
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

DB      = "/Users/muna/Hana_research/data/db/Hana_Research.db"
POP_CSV = "/Users/muna/Hana_research/data/output/transport_20261023/transport_final.csv"
OUT     = Path(f"/Users/muna/Hana_research/data/output/dict_{date.today():%Y%m%d}")
OUT.mkdir(parents=True, exist_ok=True)

ERA_CUT   = pd.Timestamp("2018-07-01")   # ARIS 定着
CTX_SENT  = 1        # アンカー文の前後何文まで
NG_MIN, NG_MAX = 3, 8
MIN_PT    = 10       # アンカー文脈での出現患者数の下限
BG_FRAC   = 0.3      # 背景文の抽出割合（計算時間対策。乱数固定）
KWIC_N    = 24       # Phase 2：1語あたりの KWIC 件数
SEED      = 2026

ARIS_RE  = re.compile(r"(?<![A-Za-z])A?RIS(?=care|[^A-Za-z]|$)|アリス\s*・?\s*ケア", re.I)
PAMPH_RE = re.compile(r"(?:看取|見取|みとり)り?の?パンフ")
# n-gram から消す文字列（アンカーそのもの）
STRIP_RE = re.compile(r"(?<![A-Za-z])A?RIS\s*(?:care|ケア)?|アリス\s*・?\s*ケア|(?:看取|見取|みとり)り?の?パンフ(?:レット)?", re.I)
# 候補から除く語（臨死期の徴候・薬剤・処置・数値）。追加・削除はここで
STOP_RE = re.compile(
    r"下顎|死前喘鳴|喘鳴|チアノーゼ|尿量|無尿|血圧|SpO|SPO|脈拍|呼吸音|努力呼吸|意識|JCS|"
    r"座薬|坐薬|座剤|坐剤|mg|ml|ブロマ|ロマゼ|ゼパム|アンヒバ|ダイアップ|モルヒネ|アンペック|オキノーム|セレネース|リスパダール|"
    r"ソルデム|ソリタ|ボルタレン|ロキソ|カロナール|フェンタ|抗生|セフ|吸引|酸素|皮下|静注|筋注|処方|"
    r"パンフ|care|ケア施行|ケア実施", re.I)
KEEP_RE  = re.compile(r"[ァ-ヶ一-龥]")          # 漢字かカタカナを含む語のみ
EDGE_RE  = re.compile(r"^[、,・/()\-:;]|[、,・/()\-:;]$")
SPLIT_RE = re.compile(r"[。\r\n]+")

DOC_VISIT_TYPES = ["定期訪問", "緊急往診", "往診", "休日往診", "夜間往診", "深夜往診", "初診"]
NURSE_TYPES     = ["訪問看護", "訪問看護(介護)", "訪問看護(緊急)"]

def nfkc(x):
    return unicodedata.normalize("NFKC", x) if isinstance(x, str) else ""

## 1. 対象・旧判定・カルテ

In [ ]:
pop = pd.read_csv(POP_CSV, dtype=str, encoding="utf-8-sig")
pop["Patient_ID"] = pop["Patient_ID"].str.strip().str.replace(r"\.0$", "", regex=True)
pop["index_date"] = pd.to_datetime(pop["index_date"], errors="coerce")

con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
old = pd.read_sql("SELECT CAST(CAST(Patient_ID AS INTEGER) AS TEXT) AS Patient_ID, unex24 FROM unexpected_death", con)
old = old.dropna(subset=["unex24"]).drop_duplicates("Patient_ID")
pop = pop.merge(old, on="Patient_ID", how="left")
print("対象:", len(pop)); print("旧判定 unex24:\n", pop["unex24"].value_counts(dropna=False))

con.execute("CREATE TEMP TABLE ids (Patient_ID TEXT PRIMARY KEY)")
con.executemany("INSERT INTO temp.ids VALUES (?)", [(p,) for p in pop["Patient_ID"]])
kt = pd.read_sql('''
    SELECT k.record_no, CAST(CAST(k.Patient_ID AS INTEGER) AS TEXT) AS Patient_ID, k.visit_type, k.visit_datetime, k.karte_text
    FROM karte k JOIN temp.ids i ON CAST(CAST(k.Patient_ID AS INTEGER) AS TEXT) = i.Patient_ID
''', con)
con.close()
kt["rec_date"] = pd.to_datetime(kt["visit_datetime"].astype(str).str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})")[0],
                                format="%Y/%m/%d", errors="coerce")
kt = kt.merge(pop[["Patient_ID", "index_date"]], on="Patient_ID")
kt = kt[kt["rec_date"] <= kt["index_date"]].copy()
kt["lead_days"] = (kt["index_date"] - kt["rec_date"]).dt.days
vt = kt["visit_type"].fillna("")
kt["source"] = np.select([vt.isin(DOC_VISIT_TYPES), vt.isin(NURSE_TYPES), vt.eq("文書")],
                         ["医師訪問", "看護", "文書"], default="その他")
kt["text_n"] = kt["karte_text"].map(nfkc)
kt["has_aris"]  = kt["text_n"].str.contains(ARIS_RE)
kt["has_pamph"] = kt["text_n"].str.contains(PAMPH_RE)
kt["anchor"]    = kt["has_aris"] | kt["has_pamph"]
print("記録:", len(kt), "/ アンカー記録:", kt["anchor"].sum())

## 2. アンカーの時期別内訳（患者数）

In [ ]:
a = kt[kt["anchor"]].copy()
a["year"] = a["rec_date"].dt.year
by_year = pd.DataFrame({
    "ARIS": a[a["has_aris"]].groupby("year")["Patient_ID"].nunique(),
    "看取りパンフ": a[a["has_pamph"]].groupby("year")["Patient_ID"].nunique(),
    "看取りパンフのみ（ARISなし記録）": a[a["has_pamph"] & ~a["has_aris"]].groupby("year")["Patient_ID"].nunique(),
}).fillna(0).astype(int)
print(by_year)
print("2018-07前にアンカーのある患者:", a.loc[a["rec_date"] < ERA_CUT, "Patient_ID"].nunique())
by_year.to_csv(OUT / "1_anchor_by_year.csv", encoding="utf-8-sig")

aris_pts = set(kt.loc[kt["has_aris"], "Patient_ID"])
pop["old_group"] = np.select(
    [pop["unex24"].isna(), pop["unex24"] == 1, pop["Patient_ID"].isin(aris_pts)],
    ["旧判定なし", "旧_予期せぬ", "旧_予期_ARISあり"], default="旧_予期_ARISなし")
print(pop["old_group"].value_counts())

## 3. 文に分割
- `ctx`：アンカー文とその前後 CTX_SENT 文
- 背景：アンカーのない記録の文
- 同じ患者の同じ文は1回

In [ ]:
rows = []
for r in kt.itertuples(index=False):
    ss = [s.strip() for s in SPLIT_RE.split(r.text_n)]
    ss = [s for s in ss if len(s) >= 2]
    anc = [i for i, s in enumerate(ss) if ARIS_RE.search(s) or PAMPH_RE.search(s)]
    ctx = {j for i in anc for j in range(i - CTX_SENT, i + CTX_SENT + 1) if 0 <= j < len(ss)}
    pre = r.rec_date < ERA_CUT
    pamph_only = r.has_pamph and not r.has_aris
    for i, s in enumerate(ss):
        rows.append((r.Patient_ID, r.record_no, i in ctx, r.anchor, pre, pamph_only, s))
sent = pd.DataFrame(rows, columns=["Patient_ID", "record_no", "ctx", "anchor_rec", "pre_era", "pamph_only", "sent"])
del rows

anc_s = (sent[sent["ctx"]].assign(sent=lambda d: d["sent"].str.replace(STRIP_RE, " ", regex=True))
         .sort_values("pre_era", ascending=False).drop_duplicates(["Patient_ID", "sent"]))
bg_s = sent[~sent["anchor_rec"]].drop_duplicates(["Patient_ID", "sent"]).sample(frac=BG_FRAC, random_state=SEED)
print("アンカー文脈の文:", len(anc_s), "/ 患者:", anc_s["Patient_ID"].nunique())
print("背景の文（抽出後）:", len(bg_s))

## 4. 候補語（Phase 1）

In [ ]:
def grams(s):
    out = set()
    for n in range(NG_MIN, NG_MAX + 1):
        for i in range(len(s) - n + 1):
            g = s[i:i + n]
            if re.search(r"\s", g) or not KEEP_RE.search(g) or EDGE_RE.search(g) or re.search(r"\d", g):
                continue
            out.add(g)
    return out

n_sent = defaultdict(int); pts = defaultdict(set); pts_pre = defaultdict(set); pts_pamph = defaultdict(set); ex = {}
for r in anc_s.itertuples(index=False):
    for g in grams(r.sent):
        n_sent[g] += 1
        pts[g].add(r.Patient_ID)
        if r.pre_era:
            pts_pre[g].add(r.Patient_ID)
        if r.pamph_only:
            pts_pamph[g].add(r.Patient_ID)
        ex.setdefault(g, r.sent[:120])

cand = pd.DataFrame({"term": list(pts)})
cand["n_pt_anchor"] = cand["term"].map(lambda g: len(pts[g]))
cand = cand[(cand["n_pt_anchor"] >= MIN_PT) & ~cand["term"].str.contains(STOP_RE)].copy()
cand["n_pt_pre2018"]  = cand["term"].map(lambda g: len(pts_pre[g]))
cand["n_pt_pamph_only"] = cand["term"].map(lambda g: len(pts_pamph[g]))
cand["n_sent_anchor"] = cand["term"].map(n_sent)

# 背景での出現（候補語のみ数える）
cset = set(cand["term"]); bg_cnt = defaultdict(int)
for s in bg_s["sent"]:
    for g in grams(s) & cset:
        bg_cnt[g] += 1
cand["n_sent_bg"] = cand["term"].map(bg_cnt).fillna(0).astype(int)
A, B = len(anc_s), len(bg_s)
a_, b_ = cand["n_sent_anchor"] + .5, cand["n_sent_bg"] + .5
cand["pct_anchor"] = (cand["n_sent_anchor"] / A * 100).round(3)
cand["pct_bg"]     = (cand["n_sent_bg"] / B * 100).round(3)
cand["log_odds"]   = np.log((a_ / (A - a_ + 1)) / (b_ / (B - b_ + 1))).round(2)

# 長い語にほぼ含まれる短い語は除く（長い語の患者数が短い語の80%以上）
npt = dict(zip(cand["term"], cand["n_pt_anchor"])); dominated = set()
for y, ny in npt.items():
    for n in range(NG_MIN, len(y)):
        for i in range(len(y) - n + 1):
            x = y[i:i + n]
            if x in npt and ny >= 0.8 * npt[x]:
                dominated.add(x)
cand = cand[~cand["term"].isin(dominated) & (cand["log_odds"] > 0)].copy()
cand["example"] = cand["term"].map(ex)
cand = cand.sort_values(["log_odds", "n_pt_anchor"], ascending=False).reset_index(drop=True)
cand.insert(0, "adopt", "")   # 1=採用候補（Phase 2 で KWIC）, 空欄=不採用
cand.to_csv(OUT / "2_candidate_terms.csv", index=False, encoding="utf-8-sig")
print("候補語:", len(cand))
pd.set_option("display.max_rows", 120, "display.width", 200)
print(cand.head(100)[["term", "n_pt_anchor", "n_pt_pre2018", "n_pt_pamph_only", "pct_anchor", "pct_bg", "log_odds"]].to_string())

---
## Phase 2：`2_candidate_terms.csv` の `adopt` 列に 1 を記入して保存後に実行
- アンカーのない記録だけから KWIC（ARIS がない書き方で使えるかを見る）
- 層：旧判定の群 × 時期（2018-07前／後）。各層からなるべく均等に抽出
- `判定` 列：採用／不採用／条件付き、`メモ` 列：理由
- `4_term_coverage.csv`：index 前30日以内に語が出た患者の割合（旧判定の群別、参考）

In [ ]:
adopt = pd.read_csv(OUT / "2_candidate_terms.csv", dtype=str, encoding="utf-8-sig")
terms = adopt.loc[adopt["adopt"].str.strip() == "1", "term"].tolist()
print("採用候補:", len(terms))

na = kt[~kt["anchor"]][["Patient_ID", "record_no", "rec_date", "lead_days", "source", "visit_type", "text_n"]]
na = na.merge(pop[["Patient_ID", "old_group"]], on="Patient_ID")
na["era"] = np.where(na["rec_date"] < ERA_CUT, "2018-07前", "2018-07以降")

rng = random.Random(SEED); kw_rows = []; cov_rows = []
for t in terms:
    hit = na[na["text_n"].str.contains(t, regex=False)]
    # KWIC：層ごとに均等（足りない層の分は他の層から）
    strata = [g.sample(frac=1, random_state=SEED) for _, g in hit.groupby(["old_group", "era"])]
    picked, i = [], 0
    while len(picked) < KWIC_N and any(i < len(s) for s in strata):
        for s in strata:
            if i < len(s) and len(picked) < KWIC_N:
                picked.append(s.iloc[i])
        i += 1
    for r in picked:
        p = r["text_n"].find(t)
        kw_rows.append({"term": t, "old_group": r["old_group"], "era": r["era"], "Patient_ID": r["Patient_ID"],
                        "record_no": r["record_no"], "source": r["source"], "visit_type": r["visit_type"],
                        "lead_days": r["lead_days"],
                        "context": r["text_n"][max(0, p - 60): p + len(t) + 60].replace("\r\n", " / ").replace("\r", " / ").replace("\n", " / ")})
    # 出現率（index 前30日以内、旧判定の群別、患者単位）
    h30 = set(hit.loc[hit["lead_days"] <= 30, "Patient_ID"])
    for g, d in pop.groupby("old_group"):
        cov_rows.append({"term": t, "old_group": g, "n": len(d), "n_hit_30d": d["Patient_ID"].isin(h30).sum()})

kwic = pd.DataFrame(kw_rows)
kwic["判定"] = ""; kwic["メモ"] = ""
kwic.to_csv(OUT / "3_kwic_review.csv", index=False, encoding="utf-8-sig")
cov = pd.DataFrame(cov_rows)
cov["pct"] = (cov["n_hit_30d"] / cov["n"] * 100).round(1)
cov = cov.pivot(index="term", columns="old_group", values="pct")
cov.to_csv(OUT / "4_term_coverage.csv", encoding="utf-8-sig")
print("KWIC:", len(kwic)); print(cov)